In [1]:
# Selecciona Income y Período para descargar: 
IncomeFile = "Ingresos2020.csv"
Periodo = 'Anual' # 'Anual' si es Resultado del Año. 'Q1'..'Q4' si es Trimestral.

### Tranformación Automatizada Datos Holded:

In [2]:
import pandas as pd 
import numpy as np 

In [3]:
# Importamos data:
    # IncomeFile = "Ingresos2022.csv"
    # Periodo = 'Anual'

Ingresos = pd.read_csv(f"/home/claude/holded/HoldedFiles/{IncomeFile}", header = None, sep = '\t') 

In [4]:
# Renombramos nombres de columnas:
Ingresos = Ingresos.rename(columns={0 : 'Subcuenta', 1 : 'Valor'})

* 0. Cabecera y pie de página:

In [5]:
# Borramos filas totalmente vacías (relleno del export de Holded):
Ingresos = Ingresos.dropna(subset=['Subcuenta', 'Valor'], how='all')

In [6]:
Ingresos

,Subcuenta,Valor
0,Tet¡xto,NaN
1,Tet¡xto,NaN
2,01/01/2020 - 31/12/2020,NaN
4,NaN,Total
5,1. Importe neto de la cifra de negocios,"3268788,014"
...,...,...
74,Impuestos sobre beneficios,"-36661,8804"
75,63000000 - IMPUESTO CORRIENTE,"-36661,8804"
76,NaN,0
77,Resultado del ejercicio,"109985,6412"


In [7]:
# Textos fijos de cabecera y pie que trae siempre el informe de Holded:
Textos_informe = pd.Series(['Tet¡xto', 'Informe creado automáticamente'])
print(Textos_informe)

0                           Tet¡xto
1    Informe creado automáticamente
dtype: str


In [8]:
# Borramos cabecera (título, rango de fechas) y pie de página:
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.contains('|'.join(Textos_informe), na=False)]
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.match(r'^\d{2}/\d{2}/\d{4}', na=False)]
Ingresos = Ingresos[Ingresos['Valor'] != 'Total']

In [9]:
Ingresos

,Subcuenta,Valor
5,1. Importe neto de la cifra de negocios,"3268788,014"
6,b) Prestaciones de servicios,"3268788,014"
7,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014"
8,NaN,0
9,4. Aprovisionamientos,"-849222,7309"
...,...,...
73,19. Impuestos sobre beneficios,"-36661,8804"
74,Impuestos sobre beneficios,"-36661,8804"
75,63000000 - IMPUESTO CORRIENTE,"-36661,8804"
76,NaN,0


* 1. Resultados (subtotales, no forman parte de la jerarquía Grupo/Subgrupo/Cuenta):

In [10]:
Resultados = pd.Series(['Resultado de explotación', 'Resultado financiero',
                         'Resultado antes de impuestos', 'Resultado del ejercicio'])
print(Resultados)

0        Resultado de explotación
1            Resultado financiero
2    Resultado antes de impuestos
3         Resultado del ejercicio
dtype: str


In [11]:
# Borramos las filas de Resultado:
Ingresos = Ingresos[~Ingresos['Subcuenta'].isin(Resultados)]
# Borramos también las filas separadoras (Subcuenta vacía, Valor = 0):
Ingresos = Ingresos[~Ingresos['Subcuenta'].isna()]

In [12]:
Ingresos

,Subcuenta,Valor
5,1. Importe neto de la cifra de negocios,"3268788,014"
6,b) Prestaciones de servicios,"3268788,014"
7,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014"
9,4. Aprovisionamientos,"-849222,7309"
10,c) Trabajos realizados por otras empresas,"-849222,7309"
11,60700000 - TRABAJOS REALIZADOS POR OTRAS,"-849222,7309"
13,6. Gastos de personal,"-217464,4149"
14,"a) Sueldos, salarios y asimilados","-200840,0112"
15,64000000 - SUELDOS Y SALARIOS,"-200840,0112"
16,b) Cargos sociales,"-16624,40371"


* 2. Grupo:

In [13]:
Grupo_id = pd.Series(['1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.', '9.', '10.',
                       '11.', '12.', '13.', '14.', '15.', '16.', '17.', '18.', '19.', '20.'])
print(Grupo_id)

0      1.
1      2.
2      3.
3      4.
4      5.
5      6.
6      7.
7      8.
8      9.
9     10.
10    11.
11    12.
12    13.
13    14.
14    15.
15    16.
16    17.
17    18.
18    19.
19    20.
dtype: str


In [14]:
Grupo_id = Grupo_id.tolist()

In [15]:
# Para crear columna Grupo:
Ingresos['Grupo'] = Ingresos['Subcuenta'].where(Ingresos['Subcuenta'].str.startswith(tuple(Grupo_id))).ffill()
# Para Borrar Filas Grupo Original:
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.startswith(tuple(Grupo_id))]

In [16]:
Ingresos

,Subcuenta,Valor,Grupo
6,b) Prestaciones de servicios,"3268788,014",1. Importe neto de la cifra de negocios
7,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014",1. Importe neto de la cifra de negocios
10,c) Trabajos realizados por otras empresas,"-849222,7309",4. Aprovisionamientos
11,60700000 - TRABAJOS REALIZADOS POR OTRAS,"-849222,7309",4. Aprovisionamientos
14,"a) Sueldos, salarios y asimilados","-200840,0112",6. Gastos de personal
15,64000000 - SUELDOS Y SALARIOS,"-200840,0112",6. Gastos de personal
16,b) Cargos sociales,"-16624,40371",6. Gastos de personal
17,64200000 - SEGURIDAD SOCIAL A CARGO DE LA,"-16624,40371",6. Gastos de personal
20,a) Servicios exteriores,"-1648052,895",7. Otros gastos de explotación
21,62300000 - SERVICIOS PROFESIONALES INDEP,"-268917,6166",7. Otros gastos de explotación


* Subgrupo:

In [17]:
# Holded no antepone letra cuando el Grupo solo tiene un único Subgrupo (ej: "Amortización 
# del inmovilizado", "Diferencias de cambio"). Normalizamos esas filas añadiendo "a) " 
# para que sigan el mismo patrón que el resto:
Subgrupos_sin_letra = pd.Series(['Amortización del inmovilizado', 'Diferencias de cambio',
                                  'Impuestos sobre beneficios'])
print(Subgrupos_sin_letra)

0    Amortización del inmovilizado
1            Diferencias de cambio
2       Impuestos sobre beneficios
dtype: str


In [18]:
Ingresos['Subcuenta'] = Ingresos['Subcuenta'].mask(Ingresos['Subcuenta'].isin(Subgrupos_sin_letra),
                                                     'a) ' + Ingresos['Subcuenta'])

In [19]:
Ingresos

,Subcuenta,Valor,Grupo
6,b) Prestaciones de servicios,"3268788,014",1. Importe neto de la cifra de negocios
7,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014",1. Importe neto de la cifra de negocios
10,c) Trabajos realizados por otras empresas,"-849222,7309",4. Aprovisionamientos
11,60700000 - TRABAJOS REALIZADOS POR OTRAS,"-849222,7309",4. Aprovisionamientos
14,"a) Sueldos, salarios y asimilados","-200840,0112",6. Gastos de personal
15,64000000 - SUELDOS Y SALARIOS,"-200840,0112",6. Gastos de personal
16,b) Cargos sociales,"-16624,40371",6. Gastos de personal
17,64200000 - SEGURIDAD SOCIAL A CARGO DE LA,"-16624,40371",6. Gastos de personal
20,a) Servicios exteriores,"-1648052,895",7. Otros gastos de explotación
21,62300000 - SERVICIOS PROFESIONALES INDEP,"-268917,6166",7. Otros gastos de explotación


In [20]:
Subgrupo_id = pd.Series(['a)', 'b)', 'c)', 'd)', 'e)', 'f)', 'g)']).tolist()

In [21]:
# Creamos columna Subgrupo:
Ingresos['SubGrupo'] = Ingresos['Subcuenta'].where(Ingresos['Subcuenta'].str.startswith(tuple(Subgrupo_id))).ffill()
# Eliminamos columna original con Subgrupo:
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.startswith(tuple(Subgrupo_id))]

In [22]:
Ingresos

,Subcuenta,Valor,Grupo,SubGrupo
7,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014",1. Importe neto de la cifra de negocios,b) Prestaciones de servicios
11,60700000 - TRABAJOS REALIZADOS POR OTRAS,"-849222,7309",4. Aprovisionamientos,c) Trabajos realizados por otras empresas
15,64000000 - SUELDOS Y SALARIOS,"-200840,0112",6. Gastos de personal,"a) Sueldos, salarios y asimilados"
17,64200000 - SEGURIDAD SOCIAL A CARGO DE LA,"-16624,40371",6. Gastos de personal,b) Cargos sociales
21,62300000 - SERVICIOS PROFESIONALES INDEP,"-268917,6166",7. Otros gastos de explotación,a) Servicios exteriores
22,62400000 - TRANSPORTES,"-744,785965",7. Otros gastos de explotación,a) Servicios exteriores
23,62500001 - SEGURO DE VIDA SABADELL,"-1047,3785",7. Otros gastos de explotación,a) Servicios exteriores
24,62600001 - Servicios bancarios SABADELL,"-1002944,625",7. Otros gastos de explotación,a) Servicios exteriores
25,62600002 - Servicios bancarios BANKIA,"-3767,71577",7. Otros gastos de explotación,a) Servicios exteriores
26,62600003 - Servicios bancarios BBVA,"-319,43982",7. Otros gastos de explotación,a) Servicios exteriores


* Cuenta: lo que queda son las cuentas hoja (código + " - " + descripción)

In [23]:
print(Ingresos)

                                            Subcuenta         Valor  \
7                70500000 - PRESTACIONES DE SERVICIOS   3268788,014   
11           60700000 - TRABAJOS REALIZADOS POR OTRAS  -849222,7309   
15                      64000000 - SUELDOS Y SALARIOS  -200840,0112   
17          64200000 - SEGURIDAD SOCIAL A CARGO DE LA  -16624,40371   
21           62300000 - SERVICIOS PROFESIONALES INDEP  -268917,6166   
22                             62400000 - TRANSPORTES   -744,785965   
23                 62500001 - SEGURO DE VIDA SABADELL    -1047,3785   
24            62600001 - Servicios bancarios SABADELL  -1002944,625   
25              62600002 - Servicios bancarios BANKIA   -3767,71577   
26                62600003 - Servicios bancarios BBVA    -319,43982   
27          62600006 - Servicios Bancarios Coinmotion     -175,9086   
28             62600007 - SERVICIOS BANCARIOS TITANES  -361411,3358   
29              62700000 - PUBLICID PROPAGANDA Y RRPP   -6626,27301   
30    

In [24]:
# Configuración Año y Periodo (Establecidos en el inicio del script)

Año = IncomeFile[8:12]

Ingresos['Año'] = Año
Ingresos['Periodo'] = Periodo

* Ordenar Columnas:

In [25]:
Ingresos = Ingresos[['Año', 'Periodo', 'Grupo', 'SubGrupo', 'Subcuenta', 'Valor']]

In [26]:
Ingresos

,Año,Periodo,Grupo,SubGrupo,Subcuenta,Valor
7,2020,Anual,1. Importe neto de la cifra de negocios,b) Prestaciones de servicios,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014"
11,2020,Anual,4. Aprovisionamientos,c) Trabajos realizados por otras empresas,60700000 - TRABAJOS REALIZADOS POR OTRAS,"-849222,7309"
15,2020,Anual,6. Gastos de personal,"a) Sueldos, salarios y asimilados",64000000 - SUELDOS Y SALARIOS,"-200840,0112"
17,2020,Anual,6. Gastos de personal,b) Cargos sociales,64200000 - SEGURIDAD SOCIAL A CARGO DE LA,"-16624,40371"
21,2020,Anual,7. Otros gastos de explotación,a) Servicios exteriores,62300000 - SERVICIOS PROFESIONALES INDEP,"-268917,6166"
22,2020,Anual,7. Otros gastos de explotación,a) Servicios exteriores,62400000 - TRANSPORTES,"-744,785965"
23,2020,Anual,7. Otros gastos de explotación,a) Servicios exteriores,62500001 - SEGURO DE VIDA SABADELL,"-1047,3785"
24,2020,Anual,7. Otros gastos de explotación,a) Servicios exteriores,62600001 - Servicios bancarios SABADELL,"-1002944,625"
25,2020,Anual,7. Otros gastos de explotación,a) Servicios exteriores,62600002 - Servicios bancarios BANKIA,"-3767,71577"
26,2020,Anual,7. Otros gastos de explotación,a) Servicios exteriores,62600003 - Servicios bancarios BBVA,"-319,43982"


* Pendiente Transformación: Desglosar Columnas correspondientes en {Id y Nombre}

In [27]:
# Division Columnas y creación columnas_id's: 

Ingresos[['Grupo_id', 'Grupo']] = Ingresos['Grupo'].str.split('. ', n = 1, expand = True)         # Grupo
Ingresos[['SubGrupo_id', 'SubGrupo']] = Ingresos['SubGrupo'].str.split(') ', n = 1, expand = True, regex = False) # SubGrupo
Ingresos[['Cuenta_id', 'Cuenta']] = Ingresos['Subcuenta'].str.split(' - ', n = 1, expand = True)   # Cuenta

In [28]:
Ingresos.head(5)

,Año,Periodo,Grupo,SubGrupo,Subcuenta,Valor,Grupo_id,SubGrupo_id,Cuenta_id,Cuenta
7,2020,Anual,Importe neto de la cifra de negocios,Prestaciones de servicios,70500000 - PRESTACIONES DE SERVICIOS,"3268788,014",1,b,70500000,PRESTACIONES DE SERVICIOS
11,2020,Anual,Aprovisionamientos,Trabajos realizados por otras empresas,60700000 - TRABAJOS REALIZADOS POR OTRAS,"-849222,7309",4,c,60700000,TRABAJOS REALIZADOS POR OTRAS
15,2020,Anual,Gastos de personal,"Sueldos, salarios y asimilados",64000000 - SUELDOS Y SALARIOS,"-200840,0112",6,a,64000000,SUELDOS Y SALARIOS
17,2020,Anual,Gastos de personal,Cargos sociales,64200000 - SEGURIDAD SOCIAL A CARGO DE LA,"-16624,40371",6,b,64200000,SEGURIDAD SOCIAL A CARGO DE LA
21,2020,Anual,Otros gastos de explotación,Servicios exteriores,62300000 - SERVICIOS PROFESIONALES INDEP,"-268917,6166",7,a,62300000,SERVICIOS PROFESIONALES INDEP


In [29]:
Ingresos.columns

Index(['Año', 'Periodo', 'Grupo', 'SubGrupo', 'Subcuenta', 'Valor', 'Grupo_id',
       'SubGrupo_id', 'Cuenta_id', 'Cuenta'],
      dtype='str')

In [30]:
# Ordenamos el dataframe: 
Ingresos = Ingresos[['Año', 'Grupo_id', 'Grupo', 'SubGrupo_id', 'SubGrupo',
                      'Cuenta_id', 'Cuenta', 'Valor']]

In [31]:
# Cambiamos tipo de dato de Valor de Object a Numeric:
Ingresos['Valor'] = Ingresos['Valor'].str.replace('.', '', regex=False)   # Elimina los puntos que separan miles
Ingresos['Valor'] = Ingresos['Valor'].str.replace(',', '.', regex=False)  # Reemplaza las comas por puntos decimales
Ingresos['Valor'] = pd.to_numeric(Ingresos['Valor'], errors='coerce')

In [32]:
# Creamos columna Valores Absolutos:
Ingresos['Abs Valor'] = Ingresos['Valor'].abs()

In [33]:
# Renombramos columnas finales para que coincidan con el resto de reportes:
Ingresos = Ingresos.rename(columns={'Grupo_id': 'ID Grupo', 'SubGrupo_id': 'ID Subgrupo',
                                     'Cuenta_id': 'ID Cuenta', 'Valor': 'Valores', 'Abs Valor': 'Abs Valores'})
Ingresos['ID Grupo'] = pd.to_numeric(Ingresos['ID Grupo'])

In [34]:
Ingresos.head(10)

,Año,ID Grupo,Grupo,ID Subgrupo,SubGrupo,ID Cuenta,Cuenta,Valores,Abs Valores
7,2020,1,Importe neto de la cifra de negocios,b,Prestaciones de servicios,70500000,PRESTACIONES DE SERVICIOS,3.268788e+06,3.268788e+06
11,2020,4,Aprovisionamientos,c,Trabajos realizados por otras empresas,60700000,TRABAJOS REALIZADOS POR OTRAS,-8.492227e+05,8.492227e+05
15,2020,6,Gastos de personal,a,"Sueldos, salarios y asimilados",64000000,SUELDOS Y SALARIOS,-2.008400e+05,2.008400e+05
17,2020,6,Gastos de personal,b,Cargos sociales,64200000,SEGURIDAD SOCIAL A CARGO DE LA,-1.662440e+04,1.662440e+04
21,2020,7,Otros gastos de explotación,a,Servicios exteriores,62300000,SERVICIOS PROFESIONALES INDEP,-2.689176e+05,2.689176e+05
22,2020,7,Otros gastos de explotación,a,Servicios exteriores,62400000,TRANSPORTES,-7.447860e+02,7.447860e+02
23,2020,7,Otros gastos de explotación,a,Servicios exteriores,62500001,SEGURO DE VIDA SABADELL,-1.047379e+03,1.047379e+03
24,2020,7,Otros gastos de explotación,a,Servicios exteriores,62600001,Servicios bancarios SABADELL,-1.002945e+06,1.002945e+06
25,2020,7,Otros gastos de explotación,a,Servicios exteriores,62600002,Servicios bancarios BANKIA,-3.767716e+03,3.767716e+03
26,2020,7,Otros gastos de explotación,a,Servicios exteriores,62600003,Servicios bancarios BBVA,-3.194398e+02,3.194398e+02


### Resultado y Descarga: 

In [35]:
Ingresos.head(4)

,Año,ID Grupo,Grupo,ID Subgrupo,SubGrupo,ID Cuenta,Cuenta,Valores,Abs Valores
7,2020,1,Importe neto de la cifra de negocios,b,Prestaciones de servicios,70500000,PRESTACIONES DE SERVICIOS,3.268788e+06,3.268788e+06
11,2020,4,Aprovisionamientos,c,Trabajos realizados por otras empresas,60700000,TRABAJOS REALIZADOS POR OTRAS,-8.492227e+05,8.492227e+05
15,2020,6,Gastos de personal,a,"Sueldos, salarios y asimilados",64000000,SUELDOS Y SALARIOS,-2.008400e+05,2.008400e+05
17,2020,6,Gastos de personal,b,Cargos sociales,64200000,SEGURIDAD SOCIAL A CARGO DE LA,-1.662440e+04,1.662440e+04


In [36]:
### Descarga arhivo: 
NombreArchivo = f"File.Income{Año}.csv"
Ingresos.to_csv(NombreArchivo, index=False)

In [37]:
f"Archivo creado llamado: 'File.Income{Año}.csv'"

"Archivo creado llamado: 'File.Income2020.csv'"